# In Class Activity April 14th, 2026

In [ ]:
pip install optuna

### Importing libraries, preparing data, initial EDA

In [2]:
# importing libraries
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


c:\Users\prchandr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# importing data
adult = pd.read_csv('c:\\Users\\prchandr\\Downloads\\adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [12]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:02 -> (00:00 left)


Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





From the EDA, I observed that the dataset has class imbalance in the target variable, with more individuals earning ≤50K than >50K, as shown in the distribution plots. I also noticed missing values in several categorical features such as workclass, occupation, and native-country. This will inform my modeling decisions by requiring me to handle missing data properly and address class imbalance, for example by tuning parameters like scale_pos_weight in XGBoost.

### Data Preprocessing (minimal) and Baseline Model

In [5]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

C:\Users\prchandr\AppData\Local\Temp\1\ipykernel_33192\2764229057.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = adult.select_dtypes(include='object').columns


,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [6]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [7]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

My baseline model is an XGBoost classifier evaluated using stratified 5-fold cross-validation. The model achieved a mean F1 score of approximately 0.712, with relatively consistent performance across folds, meaning there were no 'lucky' samples.

To improve this baseline, I plan to tune key hyperparameters of the XGBoost model. Specifically, I will explore adjusting the maximum tree depth (max_depth) to control model complexity, the learning rate (learning_rate) to manage how quickly the model updates, and scale_pos_weight to better handle class imbalance in the dataset. I will continue using stratified cross-validation to ensure fair evaluation while comparing different configurations and selecting the model that yields the highest F1 score

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [8]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
model_1 = XGBClassifier(enable_categorical=True, random_state=42)

scores_1 = cross_val_score(model_1, X, y, cv=skf, scoring='f1')
results.append(('Baseline', scores_1.mean()))

print("Baseline F1:", scores_1.mean())

# 2. MAX DEPTH
model_2 = XGBClassifier(
    max_depth=8,
    enable_categorical=True,
    random_state=42
)

scores_2 = cross_val_score(model_2, X, y, cv=skf, scoring='f1')
results.append(('Max Depth = 8', scores_2.mean()))

print("Max Depth F1:", scores_2.mean())

# 3. LEARNING RATE
model_3 = XGBClassifier(
    learning_rate=0.05,
    n_estimators=200,
    enable_categorical=True,
    random_state=42
)

scores_3 = cross_val_score(model_3, X, y, cv=skf, scoring='f1')
results.append(('Learning Rate = 0.05', scores_3.mean()))

print("Learning Rate F1:", scores_3.mean())

# 4. SCALE_POS_WEIGHT (for imbalance)
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

model_4 = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    enable_categorical=True,
    random_state=42
)

scores_4 = cross_val_score(model_4, X, y, cv=skf, scoring='f1')
results.append(('Scale Pos Weight', scores_4.mean()))

print("Scale Pos Weight F1:", scores_4.mean())


# BEST MODEL
results_df = pd.DataFrame(results, columns=['Model', 'F1 Score'])
results_df = results_df.sort_values(by='F1 Score', ascending=False)

print("\nModel Comparison:")
print(results_df)

best_model_name = results_df.iloc[0]['Model']
best_score = results_df.iloc[0]['F1 Score']

print(f"\nBest Model: {best_model_name}")
print(f"Best F1 Score: {best_score}")

Baseline F1: 0.7119671046056588
Max Depth F1: 0.7057694964103725
Learning Rate F1: 0.7108147907512212
Scale Pos Weight F1: 0.7145594081233517

Model Comparison:
                  Model  F1 Score
3      Scale Pos Weight  0.714559
0              Baseline  0.711967
2  Learning Rate = 0.05  0.710815
1         Max Depth = 8  0.705769

Best Model: Scale Pos Weight
Best F1 Score: 0.7145594081233517


### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [9]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid (4–5 hyperparameters)
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Base model
xgb = XGBClassifier(enable_categorical=True, random_state=42)

# GridSearch with stratified CV
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    verbose=1,
    n_jobs=-1
)

# Run GridSearch
grid_search.fit(X_train, y_train)

# Best parameters
print("Best Parameters:", grid_search.best_params_)

# =========================
# TRAIN FINAL MODEL
# =========================
best_model = grid_search.best_estimator_

# Evaluate on test set
y_pred = best_model.predict(X_test)

print("Test F1 Score:", f1_score(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 200, 'subsample': 1.0}
Test F1 Score: 0.720056364490371
Accuracy: 0.8779813696386529
              precision    recall  f1-score   support

           0       0.90      0.95      0.92      7431
           1       0.80      0.66      0.72      2338

    accuracy                           0.88      9769
   macro avg       0.85      0.80      0.82      9769
weighted avg       0.87      0.88      0.87      9769



### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [10]:
# tuning xgboost classifier with RandomizedSearchCV (tune more parameters than shown here)
param_dist = {
    'scale_pos_weight': np.linspace(1.0, 10.0, num=10),
    'max_depth': np.arange(3, 11),
    'learning_rate': np.linspace(0.01, 0.3, num=10)
}

# replace this placeholder model with your preferred model from above

xgb_random = RandomizedSearchCV(XGBClassifier(random_state=42, enable_categorical=True),
                                param_distributions=param_dist, n_iter=20, cv=skf, scoring='f1', random_state=42)
xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')   

# build your preferred model from above using best parameters from your RandomizedSearchCV
# and evaluate on the test set

xgb_random_best = XGBClassifier(random_state=42, scale_pos_weight=xgb_random.best_params_['scale_pos_weight'], 
                                max_depth=xgb_random.best_params_['max_depth'], 
                                learning_rate=xgb_random.best_params_['learning_rate'], 
                                enable_categorical=True)
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')


Best parameters from RandomizedSearchCV: {'scale_pos_weight': np.float64(2.0), 'max_depth': np.int64(9), 'learning_rate': np.float64(0.23555555555555557)}
Best F1 score from RandomizedSearchCV: 0.7157810076446193
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.92      0.89      0.91      7431
           1       0.69      0.76      0.72      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.83      0.81      9769
weighted avg       0.87      0.86      0.86      9769



### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [11]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    
    # replace this placeholder model with your preferred model from above
    
    xgb_optuna = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight, 
                               max_depth=max_depth,  learning_rate=learning_rate, enable_categorical=True)
    
    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=42, scale_pos_weight=study.best_params['scale_pos_weight'], 
                                  max_depth=study.best_params['max_depth'], 
                                  learning_rate=study.best_params['learning_rate'], 
                                  enable_categorical=True)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


[I 2026-04-15 23:37:23,101] A new study created in memory with name: no-name-3123a15e-8b16-4832-b1ed-10c6f61bddbf
Best trial: 0. Best value: 0.628911:   5%|▌         | 1/20 [00:09<02:53,  9.14s/it]

[I 2026-04-15 23:37:32,287] Trial 0 finished with value: 0.6289109394432959 and parameters: {'scale_pos_weight': 7.635155621465353, 'max_depth': 5, 'learning_rate': 0.03128345588627444}. Best is trial 0 with value: 0.6289109394432959.


Best trial: 1. Best value: 0.714462:  10%|█         | 2/20 [00:20<03:09, 10.54s/it]

[I 2026-04-15 23:37:43,810] Trial 1 finished with value: 0.7144622947941888 and parameters: {'scale_pos_weight': 2.967223205981711, 'max_depth': 10, 'learning_rate': 0.22888857879695715}. Best is trial 1 with value: 0.7144622947941888.


Best trial: 1. Best value: 0.714462:  15%|█▌        | 3/20 [00:31<03:03, 10.78s/it]

...

[I 2026-04-15 23:39:59,566] Trial 17 finished with value: 0.710223662661227 and parameters: {'scale_pos_weight': 3.811715121034071, 'max_depth': 7, 'learning_rate': 0.18473297912925188}. Best is trial 12 with value: 0.7251681337326473.


Best trial: 12. Best value: 0.725168:  95%|█████████▌| 19/20 [02:46<00:08,  8.75s/it]

[I 2026-04-15 23:40:09,564] Trial 18 finished with value: 0.6952166303669367 and parameters: {'scale_pos_weight': 5.237034293504882, 'max_depth': 9, 'learning_rate': 0.12003562819578693}. Best is trial 12 with value: 0.7251681337326473.


Best trial: 12. Best value: 0.725168: 100%|██████████| 20/20 [02:54<00:00,  8.73s/it]


[I 2026-04-15 23:40:17,676] Trial 19 finished with value: 0.7131865777876293 and parameters: {'scale_pos_weight': 3.5003615155606425, 'max_depth': 7, 'learning_rate': 0.1891832633660417}. Best is trial 12 with value: 0.7251681337326473.
Best parameters from Optuna: {'scale_pos_weight': 1.514396786879864, 'max_depth': 8, 'learning_rate': 0.1510111942643261}
Best F1 score from Optuna: 0.7251681337326473
Classification report for Optuna-tuned model:
              precision    recall  f1-score   support

           0       0.92      0.91      0.92      7431
           1       0.73      0.73      0.73      2338

    accuracy                           0.87      9769
   macro avg       0.82      0.82      0.82      9769
weighted avg       0.87      0.87      0.87      9769



### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


When tuning the model, I experimented with different hyperparameters such as max_depth, learning_rate, n_estimators, and scale_pos_weight. I found that adjusting these knobs had a positive impact on model performance, especially accounting for class imbalance. In terms of tuning methods, GridSearchCV was more thorough but slower, while RandomizedSearchCV was much faster and still achieved similar results. The best performance came from RandomizedSearchCV.